# Vietnamese Solar Energy RAG with LlamaIndex

Build a small retrieval pipeline for Vietnamese solar-energy documents.

This example demonstrates:
- Loading a local report with `SimpleDirectoryReader`.
- Segmenting Vietnamese text with `underthesea`.
- Embedding text with `keepitreal/vietnamese-sbert`.
- Retrieving relevant passages with `VectorStoreIndex`.

The report below is illustrative sample data, not a real energy assessment.


## 1. Install dependencies

Run this cell in a notebook environment. The embedding model may download files the first time it runs.


In [ ]:
%pip install -q llama-index-core llama-index-readers-file llama-index-embeddings-huggingface llama-index-llms-openai underthesea pypdf

## 2. Create a sample Vietnamese report

For a real project, replace this sample with your own Vietnamese `.txt` or PDF report. `SimpleDirectoryReader` supports common document formats, including PDF when the required reader dependencies are installed.


In [ ]:
from pathlib import Path

report_dir = Path("data/vietnamese_solar")
report_dir.mkdir(parents=True, exist_ok=True)

report_path = report_dir / "bao_cao_dien_mat_troi.txt"
report_path.write_text(
    """BÁO CÁO MẪU: HỆ THỐNG ĐIỆN MẶT TRỜI

1. Tổng quan
Điện mặt trời chuyển đổi năng lượng ánh sáng thành điện năng bằng các tấm pin quang điện.
Sản lượng điện thực tế phụ thuộc vào bức xạ mặt trời, hướng lắp đặt, bóng che và điều kiện thời tiết.

2. Ví dụ hệ thống giả định
Một hệ thống điện mặt trời áp mái có công suất lắp đặt giả định là 100 kWp.
Trong ví dụ minh họa này, sản lượng điện hằng năm được giả định là 150.000 kWh.
Đây là số liệu giả định để minh họa quy trình truy xuất, không phải dự báo cho một địa điểm cụ thể.

3. Các yếu tố ảnh hưởng đến sản lượng
Bụi bẩn trên bề mặt tấm pin có thể làm giảm lượng ánh sáng tiếp nhận.
Bóng che từ cây cối hoặc công trình lân cận cũng có thể làm giảm sản lượng.
Nhiệt độ, hướng và góc nghiêng của tấm pin, tổn hao dây dẫn và hiệu suất bộ biến tần đều cần được xem xét.

4. Vận hành và bảo trì
Cần kiểm tra hệ thống định kỳ, theo dõi dữ liệu của bộ biến tần và vệ sinh tấm pin khi cần thiết.
Việc đánh giá hiệu quả nên dựa trên dữ liệu vận hành thực tế và điều kiện tại địa điểm lắp đặt.
""",
    encoding="utf-8",
)

print(f"Sample report created: {report_path}")

## 3. Load the report

`SimpleDirectoryReader` reads the file and returns LlamaIndex `Document` objects with text and metadata.


In [ ]:
from llama_index.core import SimpleDirectoryReader

# Default: use the included sample report.
# To load your own PDF or TXT, change this path to your file.
input_path = report_path
# Example: input_path = Path("data/vietnamese_solar/real_report.pdf")

documents = SimpleDirectoryReader(input_files=[str(input_path)]).load_data()

if not documents:
    raise ValueError(f"No document content was loaded from {input_path}")

print(f"Loaded {len(documents)} document(s)")
print(documents[0].text[:500])

## 4. Segment Vietnamese text

Vietnamese word segmentation can help represent multi-syllable words as units. Here we use `underthesea` to preprocess the text before embedding it. The benefit depends on the corpus and retrieval task, so evaluate it on your own questions rather than assuming a fixed improvement.


In [ ]:
from llama_index.core import Document
from underthesea import word_tokenize

processed_documents = [
    Document(
        text=word_tokenize(document.text, format="text"),
        metadata=document.metadata,
    )
    for document in documents
]

print(processed_documents[0].text[:500])

## 5. Build the vector index

The Vietnamese sentence-embedding model converts document text into vectors. The first run may take some time while the model is downloaded.


In [ ]:
from llama_index.core import Settings, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="keepitreal/vietnamese-sbert"
)

index = VectorStoreIndex.from_documents(processed_documents)
print("Vector index created.")

## 6. Retrieve relevant passages

Retrieval works without configuring a generative LLM. Inspect the returned passages to see whether the retrieved context addresses the question.


In [ ]:
retriever = index.as_retriever(similarity_top_k=2)

question = "Những yếu tố nào ảnh hưởng đến sản lượng điện mặt trời?"
processed_question = word_tokenize(question, format="text")
retrieved_nodes = retriever.retrieve(processed_question)

for number, node in enumerate(retrieved_nodes, start=1):
    print(f"--- Result {number} ---")
    print(f"Similarity score: {node.score}")
    print(node.text)
    print()

## 7. Optional: generate an answer with an LLM

Retrieval above does not require an API key. To generate a natural-language answer from the retrieved context, configure an LLM. This optional example uses OpenAI and requires `OPENAI_API_KEY` to be set in the notebook environment.


In [ ]:
import os

if os.getenv("OPENAI_API_KEY"):
    from llama_index.llms.openai import OpenAI

    Settings.llm = OpenAI(model="gpt-4o-mini")
    query_engine = index.as_query_engine(similarity_top_k=2)
    response = query_engine.query(
        "Dựa trên báo cáo, những yếu tố nào ảnh hưởng đến sản lượng điện mặt trời?"
    )
    print(response)
else:
    print(
        "OPENAI_API_KEY is not set. Retrieval has been demonstrated; "
        "configure an LLM to generate an answer."
    )

## Notes

- The included report is synthetic and is intended only to demonstrate the workflow.
- For a real PDF, point `SimpleDirectoryReader` at the PDF file and inspect the extracted Vietnamese text for missing or corrupted diacritics.
- Word segmentation and embedding quality should be evaluated with representative Vietnamese queries and documents.
- The optional generation step sends retrieved context to the configured LLM provider. Review your data-handling requirements before using private documents.
